In [1]:
!pip install --user antspyx

In [2]:
import sys
import os

#####################
# Import of utils.py functions
#####################
# Required to get utils.py and access its functions
notebook_dir = os.path.abspath("")
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)
sys.path.append('.')
from utils import loadFSL, FSLeyesServer, mkdir_no_exist, interactive_MCQ

####################
# DIPY_HOME should be set prior to import of dipy to make sure all downloads point to the right folder
####################
os.environ["DIPY_HOME"] = "/home/jovyan/Data"


#############################
# Loading fsl and freesurfer within Neurodesk
# You can find the list of available other modules by clicking on the "Softwares" tab on the left
#############################
import module as lmod
await lmod.purge(force=True)
await lmod.load('fsl/6.0.7.18')
await lmod.load('freesurfer/7.4.1')
await lmod.list()

####################
# Setup FSL path
####################
loadFSL()

###################
# Load all relevant libraries for the lab
##################
import fsl.wrappers
from fsl.wrappers import fslmaths

import mne_nirs
import nilearn
from nilearn.datasets import fetch_development_fmri

import mne
import mne_nirs
import dipy
from dipy.data import fetch_bundles_2_subjects, read_bundles_2_subjects
import xml.etree.ElementTree as ET
import os.path as op
import nibabel as nib
import glob

import ants

import openneuro
from mne.datasets import sample
from mne_bids import BIDSPath, read_raw_bids, print_dir_tree, make_report

# Useful imports to define the direct download function below
import requests
import urllib.request
from tqdm import tqdm


# FSL function wrappers which we will call from python directly
from fsl.wrappers import fast, bet
from fsl.wrappers.misc import fslroi
from fsl.wrappers import flirt

# General purpose imports to handle paths, files etc
import glob
import pandas as pd
import numpy as np
import json
import subprocess

In [3]:
%gui wx


(ipykernel_launcher.py:723): dbind-WARNING **: 13:26:31.644: Couldn't connect to accessibility bus: Failed to connect to socket /run/user/203552428/at-spi/bus_100: No such file or directory


In [4]:
################
# Start FSLeyes (very neat tool to visualize MRI data of all sorts) within Python
################
fsleyesDisplay = FSLeyesServer()
fsleyesDisplay.show()

13:26:32: Debug: Adding duplicate image handler for 'Windows bitmap file'
13:26:32: Debug: Adding duplicate animation handler for '1' type
13:26:32: Debug: Adding duplicate animation handler for '2' type
13:26:32: Debug: Adding duplicate image handler for 'Windows bitmap file'
13:26:32: Debug: Adding duplicate animation handler for '1' type
13:26:32: Debug: Adding duplicate animation handler for '2' type

(ipykernel_launcher.py:723): Gtk-CRITICAL **: 13:26:33.261: gtk_window_resize: assertion 'width > 0' failed

(ipykernel_launcher.py:723): Gtk-CRITICAL **: 13:26:35.234: gtk_window_resize: assertion 'width > 0' failed


In [5]:
import os.path as op

t1_path = op.abspath("T1w.nii.gz")   
print(t1_path, op.exists(t1_path))   

/data/T1w.nii.gz True


In [6]:
def get_json_from_file(fname):
    f = open(fname)
    data = json.load(f)
    f.close()
    return data

In [7]:
t1_path = op.abspath("miniproject_2026_final/T1/T1w.nii.gz")
print(t1_path, op.isfile(t1_path))   # doit afficher True

/data/miniproject_2026_final/T1/T1w.nii.gz True


In [8]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(t1_path)


(ipykernel_launcher.py:723): Gtk-WARNING **: 13:26:46.707: Negative content width -11 (allocation 9, extents 10x10) while allocating gadget (node button, owner GtkButton)


## 1. Skull stripping

In [9]:
def get_skull_stripped_anatomical(anatomical_path, output_dir, robust=True):
    """
    Function to perform skull-stripping (removing the skull around the brain).
    This is a simple wrapper around the brain extraction tool (BET) in FSL's suite.
    It does NOT require the BIDS format.
    The method also saves the brain mask which was used to extract the brain.

    Parameters
    ----------
    anatomical_path: string
        Path to the T1w image (.nii.gz)
    output_dir: string
        Folder where the result of the brain extraction will be saved.
    robust: bool
        Whether to conduct robust center estimation with BET or not. Default is False.
    """
    os.makedirs(output_dir, exist_ok=True)
    betted_brain_path = op.join(output_dir, 'T1w')
    os.system('bet {} {} -m {}'.format(anatomical_path, betted_brain_path, '-R' if robust else ''))
    print("Done with BET.")

# paths
anatomical_path = op.abspath("miniproject_2026_final/T1/T1w.nii.gz")
output_dir = op.abspath("outputs/anat")

get_skull_stripped_anatomical(anatomical_path, output_dir)

resulting_mask_path = op.join(output_dir, 'T1w_mask')

Done with BET.


In [10]:
fsleyesDisplay.load(resulting_mask_path)

In [11]:
##################################
# Fill me with the code to use your mask
# To help you, we provide you with the skeleton of two potential approaches.
# You can fill either of them. Do not forget to test them by visualizing the result!
# For fslmaths, you can either read the documentation, or execute it without argument by running os.system('fslmaths')
##################################

# Option 1: Pythonic approach.
def apply_python_mask_approach(img_path, mask_path, masked_img_path):
    """
    The first approach, Pythonic way. The goal is, given a mask, to apply it to a T1 where the brain is to be extracted.
    
    YOU SHOULD COMPLETE THE METHOD AS IT DOES NOT WORK RIGHT NOW!

    Parameters
    ----------
    img_path: str
        Path to the image on which we would like to apply the mask (in your case, the T1 with the skull still on). Should be a .nii.gz file
    mask_path: str
        Path to the mask you would like to apply to your image. Should be a .nii.gz file, containing only binary values (0 or 1)
    masked_img_path: str
        Path to which the resulting image will be saved.
    """
    import nibabel as nib

    # Load both the T1 and the mask from disk
    img = nib.load(img_path)
    mask = nib.load(mask_path)
    
    # Load the data from both above images as numpy arrays
    img_data = img.get_fdata()
    mask_data = mask.get_fdata()

    #######################
    # Solution 1
    # Create an empty image and select all which falls in the mask (perhaps the most natural way to think about the mask)
    #######################
    # In all positions within the mask, get the image content
    #saved_img_data = np.zeros(img_data.shape)
    #saved_img_data[mask_data > 0] = img_data[mask_data > 0]

    # Save the image to disk, by creating a new Nifti image and then writing it out
    #img_out = nib.Nifti1Image(saved_img_data,img.affine, img.header)
    #nib.save(img_out, masked_img_path)

    #######################
    # Solution 2
    # Another approach is to remove from img_data all that is outside the mask.
    #######################
    
    # In all positions OUTSIDE the mask (where it is equal to 0), throw away the image
    img_data[mask_data == 0] = 0
    
    # Save the image to disk, by creating a new Nifti image and then writing it out
    img_out = nib.Nifti1Image(img_data,img.affine, img.header)
    nib.save(img_out, masked_img_path)
    
def apply_fsl_math_approach(img_path, mask_path, masked_img_path):
    ###########################
    # Solution
    # By reading fslmaths documentation, one can see that the -mas option is exactly what we desire.
    ###########################
    os.system('fslmaths {} -mas {} {}'.format(img_path, mask_path, masked_img_path))
    

anatomical_path = op.abspath("miniproject_2026_final/T1/T1w.nii.gz")# The original brain
betted_brain_path = op.join(output_dir, 'T1w_masked.nii.gz') # The brain without skull is in the derivatives folder
resulting_mask_path = op.join(output_dir, 'T1w_mask.nii.gz') # The mask to use

########################
# CHOOSE ONE OF THE TWO TO IMPLEMENT IT AND LAUNCH IT
########################
#apply_fsl_math_approach(anatomical_path, resulting_mask_path, betted_brain_path)
apply_python_mask_approach(anatomical_path, resulting_mask_path, betted_brain_path)

In [12]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(betted_brain_path)


(ipykernel_launcher.py:723): Gtk-WARNING **: 13:30:27.773: Negative content width -11 (allocation 9, extents 10x10) while allocating gadget (node button, owner GtkButton)


## 1.2 Tissue segmentation

In [ ]:
anatomical_path = op.abspath("miniproject_2026_final/T1/T1w.nii.gz")
bet_path = op.join(output_dir, 'T1w.nii.gz') 

#########################
# Solution
# By reading above: we must apply FAST to the brain-extracted image. Thus we must use the BET path brain.
##########################
fast_target = bet_path # Replace with either anatomical_path or bet_path (note: you can try both and decide which is more reasonable!)

[os.remove(f) for f in glob.glob(op.join(output_dir, '*fast*'))] # Just to clean the directory in between runs of the cell
segmentation_path = op.join(output_dir, 'T1w_fast')
fast(imgs=[fast_target], out=segmentation_path, n_classes=3)

In [14]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(bet_path)
fsleyesDisplay.load(glob.glob(op.join(output_dir, '*pve_0*'))[0])
fsleyesDisplay.load(glob.glob(op.join(output_dir, '*pve_1*'))[0])
fsleyesDisplay.load(glob.glob(op.join(output_dir, '*pve_2*'))[0])
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[1]).cmap = 'Red'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[2]).cmap = 'Green'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[3]).cmap = 'Blue'


(ipykernel_launcher.py:731): Gtk-WARNING **: 13:17:35.976: Negative content width -11 (allocation 9, extents 10x10) while allocating gadget (node button, owner GtkButton)


## 1.3 Registration

In [15]:
from fsl.wrappers import flirt

# The two images
#subject_id = '001'
subject_anatomical = op.join(output_dir, 'T1w.nii.gz')
mni_template = op.expandvars(op.join('$FSLDIR', 'data', 'standard', 'MNI152_T1_1mm_brain'))

###################
# Select which image should be source or reference
# ANSWER:
# The subject anatomical will most often be the source, as the template is usually where we want to map our subjects for 
# group comparison.
# There are cases, however, where we may want the subject anatomical to be the reference. This is the case when we want to map an 
# atlas to a subject while preserving the subject as much as possible.
# We showcase here the case where the subject is chosen as source.
#Please note that the values in the transformation matrix of flirt refer to:
#Top-left 3×3 block: rotation, scaling, shearing.
#Last column: translation in x, y, z.
#Bottom row: always [0 0 0 1] for affine transforms.
##################
source = subject_anatomical # Fill me
reference = mni_template # Fill me
result = op.join(output_dir, 'T1w_mni')
flirt(source, reference, out=result)


Final result: 
1.078909 -0.015680 0.008415 -6.129257 
0.023288 1.123096 -0.006228 -16.343431 
-0.020852 -0.011642 1.191129 -8.805752 
0.000000 0.000000 0.000000 1.000000 



{}

Flirt vizualisation : 

In [16]:
fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(reference) 
fsleyesDisplay.load(result)


(ipykernel_launcher.py:731): Gtk-WARNING **: 13:20:22.358: Negative content width -11 (allocation 9, extents 10x10) while allocating gadget (node button, owner GtkButton)


Choosing a cost:

In [29]:
#######
# Solution
# We consider only within-modalities costs, as the two images belong to the same modality: least squares and normalized correlation ratio.
# Remark correlation ratio also technically works
# So FSL FLIRT offers several cost functions for image registration. Mutual information measures statistical dependence between image intensity distributions. Correlation ratio evaluates how well one image can predict another using conditional variance statistics. Normalized correlation measures linear correlation between image intensities using the Pearson correlation coefficient. Normalized mutual information provides a scale-independent version of mutual information. Least squares minimizes squared intensity differences by assuming corresponding voxels should have similar intensities. Label difference minimizes mismatched labels for discrete segmentation images. Each function makes different assumptions about the relationship between corresponding image intensities.
#######
possible_costs = ['mutualinfo', 'corratio', 'normcorr', 'normmi', 'leastsq', 'labeldiff']
costs_to_consider = [ 'leastsq', 'normcorr' ] # fill me with the relevant costs

for c in costs_to_consider:
    flirt(source, reference, out=result + '_' + c, cost=c)


Final result: 
1.165288 -0.025935 0.016915 -13.515540 
0.029091 1.222016 -0.028761 -25.740137 
-0.015456 -0.024104 1.452098 -29.121278 
0.000000 0.000000 0.000000 1.000000 


Final result: 
1.070191 -0.015001 0.009304 -5.499236 
0.023592 1.115159 -0.005633 -15.657697 
-0.020778 -0.010821 1.178435 -7.862419 
0.000000 0.000000 0.000000 1.000000 



In [30]:
for c in costs_to_consider:
    fsleyesDisplay.load(result + '_' + c)


(ipykernel_launcher.py:5124): Gtk-WARNING **: 21:47:08.245: Negative content width -33 (allocation 1, extents 17x17) while allocating gadget (node button, owner GtkButton)

(ipykernel_launcher.py:5124): Gtk-WARNING **: 21:47:08.245: Negative content height -9 (allocation 1, extents 5x5) while allocating gadget (node button, owner GtkButton)

(ipykernel_launcher.py:5124): Gtk-WARNING **: 21:47:08.451: Negative content width -33 (allocation 1, extents 17x17) while allocating gadget (node button, owner GtkButton)

(ipykernel_launcher.py:5124): Gtk-WARNING **: 21:47:08.451: Negative content height -9 (allocation 1, extents 5x5) while allocating gadget (node button, owner GtkButton)


In [24]:
!ls -lh {output_dir}

total 35M
-rwx------ 1 jovyan jovyan  13M Oct 18 21:17 T1w.nii.gz
-rwx------ 1 jovyan jovyan 871K Oct 18 21:39 T1w_fast_mixeltype.nii.gz
-rwx------ 1 jovyan jovyan 1.7M Oct 18 21:39 T1w_fast_pve_0.nii.gz
-rwx------ 1 jovyan jovyan 2.9M Oct 18 21:39 T1w_fast_pve_1.nii.gz
-rwx------ 1 jovyan jovyan 1.6M Oct 18 21:39 T1w_fast_pve_2.nii.gz
-rwx------ 1 jovyan jovyan 759K Oct 18 21:39 T1w_fast_pveseg.nii.gz
-rwx------ 1 jovyan jovyan 755K Oct 18 21:39 T1w_fast_seg.nii.gz
-rwx------ 1 jovyan jovyan 182K Oct 18 21:17 T1w_mask.nii.gz
-rwx------ 1 jovyan jovyan  14M Oct 18 21:25 T1w_masked.nii.gz
